In [1]:
import os
import sys
# sys.path.insert(1, '')
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)

import vit_3d
import climate_dataset
import vivit
data_path = '/mnt/home/hwalters/data'
import numpy as np
import matplotlib.pyplot as plt

Current Working Directory: /Users/hayleywalters/Desktop/research 2025/ML_rain_prediction


In [2]:
def download_and_prepare_dataset(data_path):
    """Utility function to download the dataset.

    Arguments:
        data_path (string): Path to the dataset
    """
    
    with np.load(data_path, allow_pickle=True) as data:
        # Get videos
        train_videos = np.nan_to_num(data["train_images"])
        valid_videos = np.nan_to_num(data["val_images"])
        test_videos = np.nan_to_num(data["test_images"])

        # Get labels
        train_labels = data["train_labels"]#.flatten()
        valid_labels = data["val_labels"]#.flatten()
        test_labels = data["test_labels"]#.flatten()

    return (
        (train_videos, train_labels),
        (valid_videos, valid_labels),
        (test_videos, test_labels),
    )

# Get the dataset
prepared_dataset = download_and_prepare_dataset("data/cluster_1_sst.npz")
(train_videos, train_labels) = prepared_dataset[0]
(valid_videos, valid_labels) = prepared_dataset[1]
(test_videos, test_labels) = prepared_dataset[2]

print(f'train_videos {train_videos.shape}, train_labels {train_labels.shape}')
print(f'valid_videos {valid_videos.shape}, valid_labels {valid_labels.shape}')
print(f'test_videos {test_videos.shape}, test_labels {test_labels.shape}')

trainloader = vivit.prepare_dataloader(train_videos, train_labels, "train")
print("done with trainloader")
validloader = vivit.prepare_dataloader(valid_videos, valid_labels, "valid")
print("done with validloader")
testloader = vivit.prepare_dataloader(test_videos, test_labels, "test")
print("done with testloader")

train_videos (585, 24, 89, 180), train_labels (585,)
valid_videos (14, 24, 89, 180), valid_labels (14,)
test_videos (133, 24, 89, 180), test_labels (133,)
done with trainloader
done with validloader
done with testloader


In [3]:
print(vit_3d.run_experiment)  # Check if it's referencing your function
model, history = vit_3d.run_experiment(trainloader, testloader, validloader)

<function run_experiment at 0x1183228b0>
image size (89, 180) patch size (8, 4) frames 24 frame_patch_size 8
Padding height to 96 to make it divisible by 8
video shape before torch.Size([32, 24, 89, 180, 1])
pad height 7 pad width 0
Video shape after torch.Size([32, 24, 96, 180, 1])


EinopsError: Shape mismatch, can't divide axis of length 180 in chunks of 8

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

# Constants
DATA_PATH = "data/cluster_1_sst.npz"
BATCH_SIZE = 32

class VideoDataset(Dataset):
    """Dataset for loading video data."""
    
    def __init__(self, videos, labels):
        self.videos = torch.tensor(videos, dtype=torch.float32)  # Convert to float32 for efficiency
        self.labels = torch.tensor(labels, dtype=torch.long)  # Assuming classification task
        
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], self.labels[idx]

def download_and_prepare_dataset(data_path):
    """Loads dataset from .npz file and returns PyTorch DataLoader objects."""
    with np.load(data_path, allow_pickle=True) as data:
        # Load and clean data
        train_videos = np.nan_to_num(data["train_images"])
        valid_videos = np.nan_to_num(data["val_images"])
        test_videos = np.nan_to_num(data["test_images"])

        train_labels = data["train_labels"].astype(np.int64)
        valid_labels = data["val_labels"].astype(np.int64)
        test_labels = data["test_labels"].astype(np.int64)

    return train_videos, train_labels, valid_videos, valid_labels, test_videos, test_labels

def prepare_dataloader(videos, labels, batch_size=BATCH_SIZE, shuffle=True):
    """Creates a PyTorch DataLoader."""
    dataset = VideoDataset(videos, labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4, pin_memory=True)

# Usage
train_videos, train_labels, valid_videos, valid_labels, test_videos, test_labels = download_and_prepare_dataset(DATA_PATH)

print(f'train_videos {train_videos.shape}, train_labels {train_labels.shape}')
print(f'valid_videos {valid_videos.shape}, valid_labels {valid_labels.shape}')
print(f'test_videos {test_videos.shape}, test_labels {test_labels.shape}')

train_loader = prepare_dataloader(train_videos, train_labels)
print("done with trainloader")
valid_loader = prepare_dataloader(valid_videos, valid_labels, shuffle=False)
print("done with validloader")
test_loader = prepare_dataloader(test_videos, test_labels, shuffle=False)
print("done with testloader")

train_videos (585, 24, 89, 180), train_labels (585,)
valid_videos (14, 24, 89, 180), valid_labels (14,)
test_videos (133, 24, 89, 180), test_labels (133,)
done with trainloader
done with validloader
done with testloader


In [ ]:
print(vivit.run_experiment)  # Check if it's referencing your function
model, history = vivit.run_experiment(trainloader, testloader, validloader)

<function run_experiment at 0x35490df70>


NameError: name 'trainloader' is not defined